# Fuel Ticket Extraction: Fine-tuning Qwen2-VL-7B with QLoRA

**Goal:** teach an open-source vision-language model to read IOCL Fuel Delivery Vouchers and return the 24 fields as JSON.

**Pipeline (each section below is one step):**
1. Setup: connect Drive, install libraries, check the GPU
2. Configuration: every setting you may want to tune, in one place
3. Data: rebuild train/val/test from the cleaned Excel
4. Load the base model in 4-bit (QLoRA = 4-bit base model + small trainable LoRA adapters)
5. **Baseline:** accuracy of the model *before* fine-tuning on the test set
6. Fine-tune with LoRA
7. **After:** accuracy of the fine-tuned model on the same test set
8. Before vs after comparison table (for the slides)
9. Inference on new tickets → Excel

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Run cells top to bottom.
After the install cell: Runtime → Restart session, then continue from section 2.

**Key lesson from the experiments:** the vision encoder must NOT be 4-bit quantised (see section 4).
Quantising it destroyed digit-level detail and capped accuracy at ~44%; keeping it in full precision gave 88%.

## 1. Setup

In [ ]:
# Connect Google Drive so the notebook can read the Excel + images and save results.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/fuel_project'   # <- change if your folder is elsewhere
%cd {PROJECT_DIR}
!ls

In [ ]:
# Confirm a GPU is attached (should show an NVIDIA L4 with ~23 GB memory).
!nvidia-smi

In [ ]:
# Install pinned library versions known to work together for Qwen2-VL + QLoRA.
# Pinning matters: these libraries change their APIs often.
!pip install -q "transformers==4.49.0" "trl==0.15.2" "peft==0.14.0" "accelerate==1.4.0" \
                "bitsandbytes" "datasets==3.3.2" qwen-vl-utils openpyxl
# After this cell finishes: Runtime -> Restart session, then continue from section 2.

## 2. Configuration
All the settings you might tune are here. The comments say what each one does and when to change it.

In [ ]:
# (Re)connect Drive and go to the project folder - needed again after the session restart.
import os
os.chdir("/content")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
os.chdir("/content/drive/MyDrive/fuel_project")

In [ ]:
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"   # base model from Hugging Face (downloaded once, ~16 GB)

# ---- Image resolution -------------------------------------------------------
# Qwen2-VL cuts each image into 28x28-pixel patches; more pixels = more patches =
# sharper handwriting but more GPU memory and slower training.
# The processor resizes every image to fit within these limits while KEEPING the
# aspect ratio, so the ticket is never stretched or distorted.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28    # ~0.8 megapixels. If you run out of memory, lower to 768*28*28.

# ---- LoRA (the small trainable add-on) ----------------------------------------
LORA_R       = 16      # adapter size ("rank"). Higher = more capacity, more memory. Try 8-64.
LORA_ALPHA   = 32      # scaling; keep at 2 x LORA_R as a rule of thumb
LORA_DROPOUT = 0.05    # regularisation; helps prevent memorising a small dataset

# ---- Training -------------------------------------------------------------------
LEARNING_RATE = 2e-4   # standard for LoRA. If the loss jumps around, try 1e-4.
NUM_EPOCHS    = 3      # passes over the training set. With ~100 tickets, 3-5 is typical.
BATCH_SIZE    = 1      # tickets per GPU step (limited by memory)
GRAD_ACCUM    = 8      # effective batch = BATCH_SIZE x GRAD_ACCUM = 8 tickets per update

EXCEL_FILE = "fuel_tickets_labelled.xlsm"   # your cleaned labels (.xlsm or .xlsx)
DATA_DIR   = "/content/data"            # local fast-disk copy of the prepared data (see section 3)
OUTPUT_DIR = "qwen2vl-fuel-lora-visfull"   # LoRA adapter + checkpoints saved here (on Drive)

## 3. Data
Rebuilds `train/val/test.jsonl` from the cleaned Excel. The split uses a fixed seed, so it is the same every time.

In [ ]:
# Build train/val/test on Drive (skip if data/ already exists and the Excel hasn't changed),
# then copy to Colab's local disk: training reads ~100 images per epoch, and a brief Google Drive
# disconnect mid-training crashes the run ("Transport endpoint is not connected").
import os
if not os.path.exists("data/train.jsonl"):
    !python prepare_data.py --excel {EXCEL_FILE} --images images --out data
!rm -rf /content/data && cp -r data /content/data

In [ ]:
import json, os
from PIL import Image

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_rows = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_rows   = load_jsonl(f"{DATA_DIR}/val.jsonl")
test_rows  = load_jsonl(f"{DATA_DIR}/test.jsonl")
print(len(train_rows), "train |", len(val_rows), "val |", len(test_rows), "test")

PROMPT = train_rows[0]["messages"][0]["content"][1]["text"]   # the instruction used everywhere
FIELD_KEYS = list(json.loads(train_rows[0]["messages"][1]["content"][0]["text"]).keys())

def gold_answer(row):
    return json.loads(row["messages"][1]["content"][0]["text"])

def image_of(row):
    return Image.open(os.path.join(DATA_DIR, row["images"][0])).convert("RGB")

# Look at one example
display(image_of(train_rows[0]).resize((400, 520)))
print(json.dumps(gold_answer(train_rows[0]), indent=1))

## 4. Load the base model in 4-bit (the "Q" in QLoRA)
The 7B model needs ~16 GB in normal precision. Loading it in 4-bit cuts that to ~5 GB, leaving room for training on a 24 GB L4.
The base weights stay **frozen**; only the small LoRA adapters added in section 6 are trained.

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-bit format designed for neural-net weights
    bnb_4bit_use_double_quant=True,         # squeezes a little more memory
    bnb_4bit_compute_dtype=torch.bfloat16,  # maths done in 16-bit for accuracy
    # CRITICAL: keep the vision encoder (and output layer) in full precision.
    # 4-bit quantising the vision encoder blurs digit-level detail: field accuracy was capped at
    # ~44% (runs 1-2) and jumped to 88% once it was excluded (run 3). Costs ~1.5 GB extra memory.
    llm_int8_skip_modules=["visual", "lm_head"],
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Vision layer type:", type(model.visual.blocks[0].attn.qkv).__name__, "(should be Linear, not Linear4bit)")

## 5. Evaluation helpers + BASELINE (before fine-tuning)
For each test ticket the model reads the image and writes JSON, and we compare every field with the hand-checked label.
- **Field accuracy** is the % of individual fields exactly right. This is the main metric.
- **Ticket accuracy** is the % of tickets with *every* field right. This is a much stricter measure.

In [ ]:
import re, pandas as pd

def build_messages(prompt):
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]

@torch.no_grad()
def extract(model, image, max_new_tokens=600):
    """Run the model on one ticket image; return (parsed_dict_or_None, raw_text)."""
    text = processor.apply_chat_template(build_messages(PROMPT), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # greedy = repeatable
    raw = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    m = re.search(r"\{.*\}", raw, re.S)            # pull the JSON object out of the reply
    try:
        return (json.loads(m.group(0)) if m else None), raw
    except json.JSONDecodeError:
        return None, raw

def same(pred, gold):
    """Field-level match: numbers compared as numbers, text case/space-insensitive."""
    try:
        return abs(float(pred) - float(gold)) < 1e-6
    except (TypeError, ValueError):
        return str(pred).strip().upper() == str(gold).strip().upper()

def evaluate(model, rows, label):
    model.eval()
    records = []
    for i, row in enumerate(rows):
        gold = gold_answer(row)
        pred, raw = extract(model, image_of(row))
        pred = pred or {}
        rec = {"ticket": row["id"], "valid_json": bool(pred)}
        for k in FIELD_KEYS:
            rec[k] = same(pred.get(k), gold[k])
        records.append(rec)
        print(f"[{label}] {i+1}/{len(rows)} {row['id']}: {sum(rec[k] for k in FIELD_KEYS)}/{len(FIELD_KEYS)} fields correct")
    df = pd.DataFrame(records)
    field_acc = df[FIELD_KEYS].mean().mul(100).round(1)
    print(f"\n{label}: field accuracy {df[FIELD_KEYS].values.mean()*100:.1f}% | "
          f"ticket accuracy {df[FIELD_KEYS].all(axis=1).mean()*100:.1f}% | valid JSON {df.valid_json.mean()*100:.0f}%")
    return df, field_acc

In [ ]:
# BASELINE: the untrained model on the test set (~10-15 min on L4).
# If a saved baseline exists for this OUTPUT_DIR it is reloaded instead of recomputed.
csv = f"{OUTPUT_DIR}_baseline_results.csv"
if os.path.exists(csv):
    base_df = pd.read_csv(csv)
    base_acc = base_df[FIELD_KEYS].mean().mul(100).round(1)
    print(f"Loaded saved baseline: field accuracy {base_df[FIELD_KEYS].values.mean()*100:.1f}%")
else:
    base_df, base_acc = evaluate(model, test_rows, "BASELINE")
    base_df.to_csv(csv, index=False)

## 6. Fine-tune with LoRA
**What is happening:** small adapter matrices are attached to the model's attention and feed-forward layers.
Only these (~0.5% of all parameters) are trained, on our ticket→JSON examples.
The loss is computed **only on the answer tokens**, so the model learns to *produce the JSON*, not to repeat the instruction.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    # Language-model layers to adapt. (The vision encoder uses different layer names,
    # so it stays frozen; it already sees images well, the language side learns the format.)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Marker that starts the assistant's answer in Qwen's chat format; everything before it is masked.
ANSWER_START = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

def collate_fn(batch):
    """Turn a list of examples into model inputs + labels."""
    texts  = [processor.apply_chat_template(ex["messages"], tokenize=False) for ex in batch]
    images = [Image.open(os.path.join(DATA_DIR, ex["images"][0])).convert("RGB") for ex in batch]
    enc = processor(text=texts, images=images, padding=True, return_tensors="pt")

    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100                 # ignore padding
    for row in range(labels.shape[0]):                        # ignore prompt + image tokens
        ids = enc["input_ids"][row].tolist()
        for j in range(len(ids) - len(ANSWER_START), -1, -1):
            if ids[j:j + len(ANSWER_START)] == ANSWER_START:
                labels[row, : j + len(ANSWER_START)] = -100
                break
    enc["labels"] = labels
    return enc

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",          # learning rate decays smoothly to zero
    warmup_ratio=0.05,
    optim="paged_adamw_8bit",            # memory-efficient optimiser
    bf16=True,
    gradient_checkpointing=True,         # trades a little speed for a lot of memory
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    eval_strategy="epoch",               # validation loss after each epoch -> watch for overfitting
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},   # our collate_fn does the preprocessing
)

model.config.use_cache = False
trainer = SFTTrainer(
    model=model, args=args,
    train_dataset=Dataset.from_list(train_rows), eval_dataset=Dataset.from_list(val_rows),
    data_collator=collate_fn, peft_config=peft_config, processing_class=processor.tokenizer,
)
trainer.model.print_trainable_parameters()

In [ ]:
# TRAIN (~20-40 min on L4). Watch 'loss' fall; 'eval_loss' should fall too.
# If eval_loss starts RISING while training loss keeps falling, the model is memorising -> use fewer epochs.
trainer.train()

trainer.save_model(OUTPUT_DIR)          # saves only the LoRA adapter (~tens of MB) to Drive
processor.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)

In [ ]:
# Training curve for the slides
import matplotlib.pyplot as plt
log = pd.DataFrame(trainer.state.log_history)
plt.plot(log.dropna(subset=["loss"])["step"], log.dropna(subset=["loss"])["loss"], label="training loss")
if "eval_loss" in log:
    ev = log.dropna(subset=["eval_loss"]); plt.plot(ev["step"], ev["eval_loss"], "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Fine-tuning loss"); plt.show()

## 7. AFTER fine-tuning: same test set, same scoring

In [ ]:
ft_model = trainer.model
ft_model.config.use_cache = True
# After training the LoRA weights are float32 while the base model computes in bfloat16;
# autocast runs everything in bfloat16 so the two match (avoids "expected mat1 and mat2 ... same dtype").
with torch.autocast("cuda", dtype=torch.bfloat16):
    ft_df, ft_acc = evaluate(ft_model, test_rows, "FINE-TUNED")
ft_df.to_csv(f"{OUTPUT_DIR}_finetuned_results.csv", index=False)

## 8. Before vs after (the headline result)

In [ ]:
comparison = pd.DataFrame({"Baseline %": base_acc, "Fine-tuned %": ft_acc})
comparison["Change (pts)"] = (comparison["Fine-tuned %"] - comparison["Baseline %"]).round(1)
comparison.loc["OVERALL (field accuracy)"] = [
    base_df[FIELD_KEYS].values.mean()*100, ft_df[FIELD_KEYS].values.mean()*100,
    (ft_df[FIELD_KEYS].values.mean() - base_df[FIELD_KEYS].values.mean())*100]
comparison.loc["Ticket fully correct"] = [
    base_df[FIELD_KEYS].all(axis=1).mean()*100, ft_df[FIELD_KEYS].all(axis=1).mean()*100,
    (ft_df[FIELD_KEYS].all(axis=1).mean() - base_df[FIELD_KEYS].all(axis=1).mean())*100]
comparison = comparison.round(1)
comparison.to_excel("results_before_vs_after.xlsx")
comparison

## 9. Inference: extract new tickets into Excel
Put new ticket images in a folder, run this cell, and you get one Excel row per ticket.
To use the trained model later WITHOUT retraining: run sections 1-5 (helpers cell only), then:
```python
from peft import PeftModel
ft_model = PeftModel.from_pretrained(model, OUTPUT_DIR)
```

In [ ]:
import glob
NEW_DIR = "new_tickets"     # <- folder of new ticket images on Drive
rows = []
for path in sorted(glob.glob(f"{NEW_DIR}/*.jpg") + glob.glob(f"{NEW_DIR}/*.png")):
    img = Image.open(path).convert("RGB")
    img.thumbnail((1600, 1600))                      # same resizing as training data
    with torch.autocast("cuda", dtype=torch.bfloat16):
        pred, raw = extract(ft_model, img)
    rows.append({"Image_File": os.path.basename(path), **(pred or {"error": raw[:200]})})
    print(os.path.basename(path), "done")
if rows:
    pd.DataFrame(rows).to_excel("extracted_tickets.xlsx", index=False)
    print("Saved extracted_tickets.xlsx")